# Univariate Panda vs. Chronos on Full Weather (21 channels)

**Question:** Does Panda's real Panda-vs-Chronos advantage on Weather survive
when channel attention is fully suppressed at full (21-channel) scale?

**Why this experiment, and why now:** Four independent tests (Exp 9, 22, 27, 33)
found no effect of channel attention, but every one of them was either
self-referential (uni-vs-multi Panda only, no Chronos number -- Exp 9) or run on
7-channel subsets of Weather (Exp 33), never on Chronos at the full 21-channel
scale. This notebook closes that specific gap directly and cheaply
(inference-only, no retraining), reusing the harness verbatim from
`new_experiments.ipynb` and the univariate-per-channel pattern already
validated in that notebook's Burgers univariate ablation (the cell explicitly
named "Same design as Experiment 9").

**Pre-registration (fixed before running, per this project's convention):**
- **Protocol:** identical to Experiment 8 -- `H in {96, 192, 336}`,
  `n_windows = 20`, per-window instance normalisation, Wilcoxon signed-rank
  test (one-sided, Chronos > Panda), median aggregation with IQR.
- **Consistency gate:** the recomputed *multivariate* Panda-vs-Chronos numbers
  in this same session must land close to Experiment 8's logged reference
  (Panda MAE 0.6378 / Chronos MAE 0.8115 at H=96) before the univariate result
  is interpreted -- same discipline used throughout this project (e.g. B3b's
  PCA-arm consistency gate) to rule out a silent harness/session drift
  confound before trusting a new number.
- **Predicted outcome, stated before running:** given the four prior
  channel-attention nulls and univariate holding the advantage across all
  three of Exp 33's 7-channel subsets, the expected result is that univariate
  Panda still beats Chronos on full 21-channel Weather, at a magnitude
  similar to the multivariate reference (+0.17 to +0.24 MAE at Exp 8's
  H=96/192/336).
- **What would be genuinely surprising:** if the advantage collapses
  specifically at 21 channels despite holding at 7-channel subsets -- this
  would implicate channel *count* as a variable none of the four prior tests
  actually varied (all stayed at <=7 channels or were self-referential).

**Horizons above 128:** Panda's native prediction horizon is 128 steps; H=192
and H=336 require the same sliding-window autoregressive rollout used by the
standard `panda_forecast()`. The univariate function below applies that same
rollout **per channel independently**, so multi-horizon univariate forecasts
are handled consistently with the multivariate baseline rather than only
being valid at H<=128 (a gap in the existing Burgers univariate ablation cell,
which was never exercised past H=96/192-equivalent PRED_LEN values).


## Setup -- imports, model loading, and the exact harness from `new_experiments.ipynb`

In [1]:
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

N_WINDOWS   = 20
CONTEXT_LEN = 512
PRED_LEN    = 96          # default reference horizon; the sweep below overrides per-call
DATA_DIR    = './ts_data' # adjust if your data lives elsewhere


Device: cpu


In [2]:
import sys
sys.path.insert(0, './panda')

from panda.patchtst.pipeline import PatchTSTPipeline
from chronos import ChronosPipeline

panda_model = PatchTSTPipeline.from_pretrained(
    mode='predict',
    pretrain_path='GilpinLab/panda',
    device_map=device,
)

chronos_model = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-small',
    device_map=device,
    torch_dtype=torch.bfloat16,
)

print('Models loaded.')


Models loaded.


In [3]:
# -------------------------------------------------------
# Metrics
# -------------------------------------------------------
def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

# -------------------------------------------------------
# Per-window normalisation
# -------------------------------------------------------
def instance_norm_window(x_CT):
    """x_CT: (C, T). Normalise per channel using this window only."""
    mu  = x_CT.mean(axis=1, keepdims=True)
    std = x_CT.std( axis=1, keepdims=True) + 1e-8
    return (x_CT - mu) / std, mu, std

def load_ts(path):
    """Raw (C, T) -- no global normalisation."""
    df = pd.read_csv(path)
    df = df.select_dtypes(include=[np.number])
    return df.values.astype(np.float32).T  # (C, T)

# -------------------------------------------------------
# Inference -- verbatim from new_experiments.ipynb Cell 3
# -------------------------------------------------------
def panda_forecast(context_np, horizon):
    """context_np: (C, T) normalised. Returns (C, horizon). Multivariate (standard)."""
    TRAIN_H   = 128
    remaining = horizon
    ctx       = context_np.copy()
    preds     = []
    while remaining > 0:
        h         = min(TRAIN_H, remaining)
        context_t = torch.tensor(ctx.T, dtype=torch.float32)
        with torch.no_grad():
            pred = panda_model.predict(
                context_t, h,
                limit_prediction_length=False,
                sliding_context=True,
            )
        p = pred.squeeze().cpu().numpy()
        if p.ndim == 1:
            p = p[:, None]
        if p.shape[0] != context_np.shape[0]:
            p = p.T
        preds.append(p[:, :h])
        ctx       = np.concatenate([ctx[:, h:], p[:, :h]], axis=1)
        remaining -= h
    return np.concatenate(preds, axis=1)  # (C, horizon)

def chronos_forecast(context_np, horizon):
    """Batched -- all channels in one call."""
    ctx = torch.tensor(context_np, dtype=torch.float32)
    with torch.no_grad():
        out = chronos_model.predict(
            ctx, prediction_length=horizon, num_samples=1
        )
    return out[:, 0, :].cpu().numpy()  # (C, horizon)

# -------------------------------------------------------
# Core evaluator -- verbatim from new_experiments.ipynb Cell 3
# -------------------------------------------------------
def evaluate(data_CT, horizon, n_windows=N_WINDOWS, label='',
             fn_a=None, fn_b=None,
             name_a='panda', name_b='chronos'):
    """
    data_CT: (C, T) RAW. Normalises each window independently.
    fn_a, fn_b: (context_normed: (C,T), horizon) -> (C, H)
    """
    if fn_a is None: fn_a = panda_forecast
    if fn_b is None: fn_b = chronos_forecast

    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    if max_start <= 0:
        print(f'  [SKIP] {label}: T={T} too short')
        return None

    starts = np.linspace(0, max_start, n_windows, dtype=int)
    mae_a, mae_b = [], []

    for s in starts:
        ctx_raw           = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw           = data_CT[:, s + CONTEXT_LEN : s + CONTEXT_LEN + horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm          = (tgt_raw - mu) / std
        mae_a.append(mae(tgt_norm, fn_a(ctx_norm, horizon)))
        mae_b.append(mae(tgt_norm, fn_b(ctx_norm, horizon)))

    diff = np.array(mae_b) - np.array(mae_a)
    try:
        _, pval = wilcoxon(diff, alternative='greater') \
            if np.any(diff != 0) else (0, 1.0)
    except Exception:
        pval = np.nan

    adv = np.median(mae_b) - np.median(mae_a)
    sig = ' *' if pval < 0.05 else (' ~' if pval < 0.10 else '')
    iqr_a = np.percentile(mae_a,75) - np.percentile(mae_a,25)
    iqr_b = np.percentile(mae_b,75) - np.percentile(mae_b,25)

    result = {
        'label'         : label,
        'horizon'       : horizon,
        'name_a'        : name_a,
        'name_b'        : name_b,
        f'{name_a}_mae' : np.median(mae_a),
        f'{name_a}_iqr' : iqr_a,
        f'{name_b}_mae' : np.median(mae_b),
        f'{name_b}_iqr' : iqr_b,
        'advantage_mae' : adv,
        'wilcoxon_p'    : pval,
    }
    print(
        f'  {label:50s}  H={horizon:4d}  '
        f'{name_a}={np.median(mae_a):.4f}[+/-{iqr_a:.4f}]  '
        f'{name_b}={np.median(mae_b):.4f}[+/-{iqr_b:.4f}]  '
        f'Adv={adv:+.4f}  p={pval:.3f}{sig}'
    )
    return result

print('Harness defined (verbatim from new_experiments.ipynb).')


Harness defined (verbatim from new_experiments.ipynb).


## Univariate forecast function -- full 21-channel Weather, with multi-horizon rollout

In [4]:
def panda_forecast_univariate(context_np, horizon):
    """
    Each channel processed independently through Panda -- suppresses
    cross-channel attention entirely. Applies the same sliding-window
    autoregressive rollout as panda_forecast() (TRAIN_H=128 chunks) so that
    H=192 and H=336 are handled correctly per channel, not just H<=128.
    context_np: (C, T) normalised. Returns (C, horizon).
    """
    TRAIN_H = 128
    C = context_np.shape[0]
    all_preds = []
    for c in range(C):
        remaining = horizon
        ctx = context_np[c:c+1, :].copy()  # (1, T)
        ch_preds = []
        while remaining > 0:
            h = min(TRAIN_H, remaining)
            context_t = torch.tensor(ctx.T, dtype=torch.float32)
            with torch.no_grad():
                pred = panda_model.predict(
                    context_t, h,
                    limit_prediction_length=False,
                    sliding_context=True,
                )
            p = pred.squeeze().cpu().numpy()
            if p.ndim == 0:
                p = np.array([float(p)])
            if p.ndim == 1:
                p = p[None, :]        # (1, h)
            if p.shape[0] != 1:
                p = p.T
            p = p[:, :h]
            ch_preds.append(p)
            ctx = np.concatenate([ctx[:, h:], p], axis=1)
            remaining -= h
        all_preds.append(np.concatenate(ch_preds, axis=1)[0])  # (horizon,)
    return np.stack(all_preds, axis=0)  # (C, horizon)

print('panda_forecast_univariate defined (per-channel, multi-horizon-safe).')


panda_forecast_univariate defined (per-channel, multi-horizon-safe).


## Load Weather and run the consistency gate (multivariate reference vs. Experiment 8)

In [5]:
data_weather = load_ts(f'{DATA_DIR}/weather.csv')
print(f'Weather shape: {data_weather.shape}')
assert data_weather.shape[0] == 21, f'Expected 21 channels, got {data_weather.shape[0]}'


Weather shape: (21, 52696)


In [6]:
# Consistency gate: recompute the standard multivariate result at H=96 in this
# session and compare against Experiment 8's logged reference before trusting
# the univariate result below. Reference: Panda MAE 0.6378, Chronos MAE 0.8115
# (Exp 8, n=20, H=96).

EXP8_REFERENCE = {'panda_mae': 0.6378, 'chronos_mae': 0.8115}
GATE_TOL = 0.05  # absolute MAE tolerance, generous given known session-to-session drift

r_multi_gate = evaluate(data_weather, 96, n_windows=N_WINDOWS, label='Weather_multi_H96_gate')

panda_diff   = abs(r_multi_gate['panda_mae']   - EXP8_REFERENCE['panda_mae'])
chronos_diff = abs(r_multi_gate['chronos_mae'] - EXP8_REFERENCE['chronos_mae'])

print(f"\nConsistency gate vs Experiment 8 reference:")
print(f"  Panda:   this session={r_multi_gate['panda_mae']:.4f}  "
      f"reference={EXP8_REFERENCE['panda_mae']:.4f}  diff={panda_diff:.4f}")
print(f"  Chronos: this session={r_multi_gate['chronos_mae']:.4f}  "
      f"reference={EXP8_REFERENCE['chronos_mae']:.4f}  diff={chronos_diff:.4f}")

GATE_PASSED = (panda_diff <= GATE_TOL) and (chronos_diff <= GATE_TOL)
print(f"\nGATE_PASSED = {GATE_PASSED}")
if not GATE_PASSED:
    print('  WARNING: this session diverges from Experiment 8 by more than the '
          'tolerance. Investigate before trusting the univariate result below '
          '(model version drift, data file mismatch, or a harness change are '
          'the likely candidates) -- do not silently proceed on a failed gate.')


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


KeyboardInterrupt: 

## Main run -- univariate Panda vs. Chronos, full 21-channel Weather, H in {96, 192, 336}

In [ ]:
HORIZONS = [96, 192, 336]
uni_results = []

print('Univariate Panda vs Chronos -- full 21-channel Weather')
print('-' * 70)

for h in HORIZONS:
    r_uni = evaluate(
        data_weather, h, n_windows=N_WINDOWS,
        label=f'Weather_univariate_H{h}',
        fn_a=panda_forecast_univariate, fn_b=chronos_forecast,
        name_a='panda_uni', name_b='chronos',
    )
    if r_uni:
        uni_results.append(r_uni)

df_uni = pd.DataFrame(uni_results)
df_uni.to_csv('weather_univariate_advantage.csv', index=False)
print('\nSaved weather_univariate_advantage.csv')


## Reference -- recompute multivariate at all three horizons for direct within-session comparison

In [ ]:
multi_results = [r_multi_gate]  # H=96 already computed above (the gate run)

for h in [192, 336]:
    r_multi = evaluate(data_weather, h, n_windows=N_WINDOWS, label=f'Weather_multivariate_H{h}')
    if r_multi:
        multi_results.append(r_multi)

df_multi = pd.DataFrame(multi_results)
df_multi.to_csv('weather_multivariate_reference.csv', index=False)
print('\nSaved weather_multivariate_reference.csv')


## Summary -- univariate vs. multivariate advantage, side by side

In [ ]:
print('=' * 90)
print('SUMMARY: Panda advantage over Chronos, full 21-channel Weather')
print('=' * 90)
print(f"{'H':>5} | {'multi_panda':>11} | {'multi_adv':>10} | {'multi_p':>8} | "
      f"{'uni_panda':>10} | {'uni_adv':>9} | {'uni_p':>7}")
print('-' * 90)

for h in HORIZONS:
    m = df_multi[df_multi.horizon == h].iloc[0]
    u = df_uni[df_uni.horizon == h].iloc[0]
    print(f"{h:>5} | {m['panda_mae']:>11.4f} | {m['advantage_mae']:>+10.4f} | "
          f"{m['wilcoxon_p']:>8.4f} | {u['panda_uni_mae']:>10.4f} | "
          f"{u['advantage_mae']:>+9.4f} | {u['wilcoxon_p']:>7.4f}")

print()
print('Interpretation guide (per pre-registration above):')
print('  - If uni_adv is positive and significant (p<0.05) at all three horizons,')
print('    at similar magnitude to multi_adv: prediction confirmed, channel')
print('    attention is not required for the Weather advantage at full scale.')
print('  - If uni_adv collapses specifically relative to the 7-channel subset')
print('    results (Exp 33) despite holding here: no new finding, consistent')
print('    with the existing four nulls.')
print('  - If uni_adv collapses at 21 channels in a way NOT seen at 7 channels:')
print('    genuinely new and surprising -- channel count itself would be')
print('    implicated as a variable, distinct from channel attention per se.')
